# Predict Customer Churn — LightGBM Tuned Multi-Seed + Feature Engineering

LightGBM equivalent of notebook 06 (XGBoost), with feature engineering added.

1. **Feature Engineering** — same row-stat + domain interaction features as XGBoost pipeline
2. **Robust Optuna Tuning** — `lgb.cv()` inside each trial, dynamic `num_leaves` cap
3. **Multi-Seed Ensemble** — 5 seeds × 10 folds = 50 models
4. **GPU Support** — `device_type='gpu'` auto-injected when GPU detected
5. **Progress Bars** — `tqdm` on all long loops

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna
from tqdm.auto import tqdm
import warnings
import subprocess
import os
import gc

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment Detection ─────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')

def has_local_gpu():
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False

USE_GPU  = IS_KAGGLE or has_local_gpu()
DATA_DIR = '/kaggle/input/playground-series-s6e3/' if IS_KAGGLE else '../data/'
SUB_DIR  = '/kaggle/working/'                       if IS_KAGGLE else '../'

print(f"Environment  : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"GPU          : {'Enabled ✓' if USE_GPU else 'Disabled — using CPU'}")
print(f"Data dir     : {DATA_DIR}")
print(f"Output dir   : {SUB_DIR}")
print(f"LightGBM ver : {lgb.__version__}")

# ── Experiment Settings ───────────────────────────────────────────────────────
RUN_TUNING   = True
N_TRIALS     = 50     # 100 → 50: avoids Kaggle 9-hr GPU limit (tuning+ensemble ~4-5 hrs)
N_CV_FOLDS   = 5
SEEDS        = [42, 2024, 777, 1337, 999]
N_SPLITS     = 10
TOTAL_MODELS = len(SEEDS) * N_SPLITS

# Pre-computed fallback (used when RUN_TUNING = False)
PRECOMPUTED_PARAMS = {
    'learning_rate':    0.01811,
    'num_leaves':       84,
    'max_depth':        10,
    'feature_fraction': 0.6537,
    'min_child_samples': 20,
    'lambda_l1':        0.01859,
    'lambda_l2':        0.00646,
}

## 1. Feature Engineering

In [ ]:
def engineer_features(df):
    """Row-level statistics + domain interaction features on numeric columns."""
    df = df.copy()
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

    df['num_sum']  = df[num_cols].sum(axis=1)
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    df['Average_Monthly_Actual'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Monthly_diff']           = df['MonthlyCharges'] - df['Average_Monthly_Actual']
    df['Monthly_ratio']          = df['MonthlyCharges'] / (df['Average_Monthly_Actual'] + 1e-5)

    return df

print('Feature engineering function defined.')

## 2. Load & Preprocess Data

In [ ]:
print(f'Loading data from {DATA_DIR} ...')
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub      = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

TARGET = 'Churn'
if train_df[TARGET].dtype == 'object':
    train_df[TARGET] = train_df[TARGET].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

train_df['is_train'] = 1
test_df['is_train']  = 0
df = pd.concat([train_df, test_df], ignore_index=True)

features = [c for c in train_df.columns if c not in ['id', TARGET, 'is_train']]
cat_features = []

print('Label encoding categorical features...')
for col in tqdm(features, desc='Encoding'):
    if df[col].dtype == 'object':
        cat_features.append(col)
        le = LabelEncoder()
        df[col] = df[col].fillna('Missing')
        df[col] = le.fit_transform(df[col].astype(str))

num_features = [c for c in features if c not in cat_features]
for col in num_features:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

train_enc = df[df['is_train'] == 1].reset_index(drop=True)
test_enc  = df[df['is_train'] == 0].reset_index(drop=True)

print('\nEngineering features...')
X      = engineer_features(train_enc[features])
X_test = engineer_features(test_enc[features])
y      = train_enc[TARGET]

print(f'\nTrain shape  : {X.shape}')
print(f'Test shape   : {X_test.shape}')
print(f'Target dist  : {y.value_counts().to_dict()}')
print(f'New features : {[c for c in X.columns if c not in features]}')

## 3. Optuna Hyperparameter Tuning

`lgb.cv()` runs a full stratified CV **per trial**. `num_leaves` is dynamically capped at `2^max_depth - 1` to prevent invalid configurations.

In [ ]:
base_params = {
    'objective':          'binary',
    'metric':             'auc',
    'boosting_type':      'gbdt',
    'verbose':            -1,
    'n_jobs':             -1,
    'feature_pre_filter': False,
}
if USE_GPU:
    base_params['device_type']     = 'gpu'
    base_params['gpu_platform_id'] = 0
    base_params['gpu_device_id']   = 0
    print('GPU params injected: device_type=gpu, platform=0, device=0')

# Build full dataset once — reused across all Optuna trials
dtrain_full = lgb.Dataset(
    X, label=y,
    categorical_feature=cat_features,
    free_raw_data=False,
)

if RUN_TUNING:
    print(f'\nStarting Optuna tuning: {N_TRIALS} trials × {N_CV_FOLDS}-fold CV each')

    pbar = tqdm(total=N_TRIALS, desc='Optuna Trials', unit='trial')
    best_value_so_far = [0.0]

    def objective(trial):
        max_depth  = trial.suggest_int('max_depth', 3, 10)
        max_leaves = min(127, 2 ** max_depth - 1)
        # ── FIX: clamp low so it never exceeds max_leaves ──────────────────
        leaves_low = min(15, max_leaves)

        params = {
            **base_params,
            'learning_rate':     trial.suggest_float('learning_rate',     0.005, 0.1,  log=True),
            'num_leaves':        trial.suggest_int(  'num_leaves',        leaves_low, max_leaves),
            'max_depth':         max_depth,
            'feature_fraction':  trial.suggest_float('feature_fraction',  0.4,   1.0),
            'bagging_fraction':  trial.suggest_float('bagging_fraction',  0.4,   1.0),
            'bagging_freq':      trial.suggest_int(  'bagging_freq',      1,     7),
            'min_child_samples': trial.suggest_int(  'min_child_samples', 5,     100),
            'lambda_l1':         trial.suggest_float('lambda_l1',         1e-8,  10.0, log=True),
            'lambda_l2':         trial.suggest_float('lambda_l2',         1e-8,  10.0, log=True),
        }

        try:
            cv_result = lgb.cv(
                params,
                dtrain_full,
                num_boost_round   = 2000,
                nfold             = N_CV_FOLDS,
                stratified        = True,
                callbacks         = [lgb.early_stopping(50, verbose=False)],
                seed              = 42,
                return_cvbooster  = False,
            )
            score     = max(cv_result['valid auc-mean'])
            best_iter = int(np.argmax(cv_result['valid auc-mean']))
            trial.set_user_attr('best_iteration', best_iter)
        except lgb.basic.LightGBMError:
            raise optuna.TrialPruned()

        if score > best_value_so_far[0]:
            best_value_so_far[0] = score
        pbar.set_postfix({'AUC': f'{score:.5f}', 'Best': f'{best_value_so_far[0]:.5f}'})
        pbar.update(1)
        return score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_trial = study.best_trial
    best_iter  = best_trial.user_attrs.get('best_iteration', 500)

    print(f"\n{'='*55}")
    print(f"Best CV AUC    : {best_trial.value:.5f}")
    print(f"Best iteration : {best_iter}")
    print(f"Best params    :")
    for k, v in best_trial.params.items():
        print(f"  {k:25s}: {v}")
    print(f"{'='*55}")

    best_params = {**base_params, **best_trial.params}

    results_path = os.path.join(SUB_DIR, 'lgbm_tuning_results.csv')
    study.trials_dataframe().to_csv(results_path, index=False)
    print(f'\nAll trial results saved → {results_path}')

else:
    print('Skipping tuning — using PRECOMPUTED_PARAMS.')
    best_params = {**base_params, **PRECOMPUTED_PARAMS}
    best_iter   = 500
    for k, v in PRECOMPUTED_PARAMS.items():
        print(f'  {k:25s}: {v}')

## 4. Multi-Seed Ensemble

In [ ]:
print(f'Multi-Seed Ensemble: {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS} models\n')

test_preds_sum = np.zeros(len(X_test))
global_oof_sum = np.zeros(len(X))
fold_auc_log   = []

outer_bar = tqdm(SEEDS, desc='Seeds', position=0)

for seed in outer_bar:
    outer_bar.set_description(f'Seed {seed}')
    skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X))

    inner_bar = tqdm(
        enumerate(skf.split(X, y)),
        total    = N_SPLITS,
        desc     = '  Folds',
        position = 1,
        leave    = False,
    )

    for fold, (train_idx, val_idx) in inner_bar:
        X_tr,  X_val  = X.iloc[train_idx],  X.iloc[val_idx]
        y_tr,  y_val  = y.iloc[train_idx],  y.iloc[val_idx]

        lgb_train = lgb.Dataset(
            X_tr, label=y_tr,
            categorical_feature=cat_features,
            free_raw_data=False,
        )
        lgb_val = lgb.Dataset(
            X_val, label=y_val,
            reference=lgb_train,
            categorical_feature=cat_features,
            free_raw_data=False,
        )

        fold_params = {**best_params, 'seed': seed, 'bagging_seed': seed}

        model = lgb.train(
            fold_params,
            lgb_train,
            num_boost_round = best_iter + 200,
            valid_sets      = [lgb_val],
            callbacks       = [lgb.early_stopping(150, verbose=False)],
        )

        val_preds  = model.predict(X_val,  num_iteration=model.best_iteration)
        test_preds = model.predict(X_test, num_iteration=model.best_iteration)

        fold_auc = roc_auc_score(y_val, val_preds)
        fold_auc_log.append(fold_auc)

        seed_oof[val_idx]        = val_preds
        global_oof_sum[val_idx] += val_preds
        test_preds_sum          += test_preds

        inner_bar.set_postfix({'fold_auc': f'{fold_auc:.5f}', 'best_iter': model.best_iteration})

        del model, lgb_train, lgb_val
        gc.collect()

    seed_auc = roc_auc_score(y, seed_oof)
    outer_bar.set_postfix({'seed_oof_auc': f'{seed_auc:.5f}'})

# ── Final metrics ─────────────────────────────────────────────────────────────
global_oof = global_oof_sum / len(SEEDS)
final_auc  = roc_auc_score(y, global_oof)

print(f"\n{'='*55}")
print(f"Fold AUC — mean : {np.mean(fold_auc_log):.5f}  std: {np.std(fold_auc_log):.5f}")
print(f"Global Ensemble OOF AUC : {final_auc:.5f}")
print(f"{'='*55}")

## 5. Generate Submission

In [ ]:
final_test_preds = test_preds_sum / TOTAL_MODELS
sub[TARGET] = final_test_preds

out_file = os.path.join(SUB_DIR, 'submission_lgbm_tuned_multiseed_fe.csv')
sub.to_csv(out_file, index=False)

print(f'Submission saved → {out_file}')
print(f'Prediction range : [{final_test_preds.min():.4f}, {final_test_preds.max():.4f}]')
display(sub.head())